# Educational model
___________________________________________
## Transformer Model

In [265]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense
from tensorflow.keras.layers import MultiHeadAttention
from tensorflow.keras.layers import LayerNormalization
from tensorflow.keras.layers import GlobalAveragePooling1D

In [267]:
texts = [
    "i love this movie",
    "this film is amazing",
    "i hate this movie",
    "this film is bad"
]

labels = [1, 1, 0, 0]  # 1 = positive, 0 = negative

In [312]:
tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(texts)

X = tokenizer.texts_to_sequences(texts)

X = pad_sequences(X, maxlen=10)
y = labels

### Build Model

In [314]:
max_len = 10
vocab_size = 1000
embed_dim = 16
num_heads = 2
ff_dim = 16

inputs = Input(shape=(max_len,))

# Embedding
x = Embedding(vocab_size, embed_dim)(inputs)

# Self-Attention
attn_output = MultiHeadAttention(
    num_heads=num_heads,
    key_dim=embed_dim
)(x, x)

# Add & Norm
x = LayerNormalization()(x + attn_output)

# Feed Forward
ff = Dense(ff_dim, activation="relu")(x)
ff = Dense(embed_dim)(ff)

# Add & Norm
x = LayerNormalization()(x + ff)

# Pooling
x = GlobalAveragePooling1D()(x)

# Output
outputs = Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

### Training Model

In [322]:
import numpy as np

X = np.array(X, dtype=np.int32)
y = np.array(y, dtype=np.float32)

print(type(X), X.shape)
print(type(y), y.shape)

history = model.fit(X, y, epochs=20, batch_size=2)

<class 'numpy.ndarray'> (4, 10)
<class 'numpy.ndarray'> (4,)
Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 46ms/step - accuracy: 0.5000 - loss: 1.0615
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.3333 - loss: 0.7534
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.8333 - loss: 0.6754
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.5000 - loss: 0.7489
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.5000 - loss: 0.7280
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6667 - loss: 0.6875
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6667 - loss: 0.6600
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.6667 - loss: 0.6528
Epoch 9/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.5000 - loss: 0.6574
Epoch 10/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5000 - loss: 0.6448
Epoch 11/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.6667 - loss: 0.5920
Epoch 12/20
2/2 ━━━━━━━━

### prediction

In [340]:
test_texts = ["this movie is bad", "this film is amazing"]

test_seq = tokenizer.texts_to_sequences(test_texts)
test_pad = pad_sequences(test_seq, maxlen=10)

pred = model.predict(test_pad)
print(np.round(pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
[[0.]
 [1.]]
